In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings

# Suppress warnings from statsmodels to keep the output clean
warnings.filterwarnings("ignore")

# --- 1. Configuration ---
input_file = '../../data/data_with_features.csv'
output_file = '../../data/syntheticData/synthetic_data_sarima_statsmodels.csv'
target_column = 'cumulative_mite_count'
group_column = 'treatment_id'

# --- 2. Load the Original Data ---
try:
    df = pd.read_csv(input_file)
    print(f"✅ Successfully loaded original data from '{input_file}'. Shape: {df.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file '{input_file}' was not found.")
    exit()

# --- 3. Grid Search and SARIMA Generation ---
print(f"\n⚙️  Starting grouped SARIMA generation using `statsmodels`...")

# Define a small grid of common (p,d,q) parameters to search over
# (p=autoregressive, d=differencing, q=moving average)
param_grid = [(1, 1, 1), (1, 1, 0), (0, 1, 1), (1, 0, 0), (0, 0, 1)]
# Define the seasonal order (P,D,Q,m). We'll keep this fixed for simplicity.
# (1,1,0,7) is a common choice for weekly seasonality on daily data.
seasonal_order = (1, 1, 0, 7)

all_synthetic_groups = []
grouped_data = df.groupby(group_column)
num_groups = len(grouped_data)
print(f"   Found {num_groups} unique time series to process.")

# Process each group
for i, (group_name, group_df) in enumerate(grouped_data):
    print(f"   Processing group {i+1}/{num_groups} (ID: {group_name})... ", end="")
    group_df = group_df.reset_index(drop=True)
    original_series = group_df[target_column]

    if len(original_series) < 20:
        print("Series too short, copying original data.")
        all_synthetic_groups.append(group_df)
        continue

    best_aic = np.inf
    best_order = None
    
    # Grid search for the best (p,d,q) order
    for order in param_grid:
        try:
            model = SARIMAX(original_series, order=order, seasonal_order=seasonal_order)
            results = model.fit(disp=False)
            if results.aic < best_aic:
                best_aic = results.aic
                best_order = order
        except Exception:
            continue
            
    # Fit the best model and generate data
    if best_order:
        try:
            best_model = SARIMAX(original_series, order=best_order, seasonal_order=seasonal_order)
            best_results = best_model.fit(disp=False)
            
            # Get in-sample predictions to create the synthetic series
            synthetic_series = best_results.get_prediction(start=0, end=len(original_series)-1).predicted_mean
            
            # --- CRITICAL: Enforce Constraints ---
            synthetic_series = np.maximum(0, synthetic_series) # Ensure non-negativity
            synthetic_series = np.maximum.accumulate(synthetic_series) # Ensure non-decreasing

            synthetic_group_df = group_df.copy()
            synthetic_group_df[target_column] = synthetic_series
            all_synthetic_groups.append(synthetic_group_df)
            print(f"Best order {best_order} found. Data generated.")
        except Exception:
            print("Failed to fit best model, copying original data.")
            all_synthetic_groups.append(group_df)
    else:
        print("No suitable model found, copying original data.")
        all_synthetic_groups.append(group_df)

# Combine all synthetic groups into a final DataFrame
final_synthetic_df = pd.concat(all_synthetic_groups, ignore_index=True)

# --- 4. Save the Synthetic Data ---
final_synthetic_df.to_csv(output_file, index=False)

print("\n✅ Process complete!")
print(f"   Synthetic data generated via `statsmodels` SARIMAX and saved to '{output_file}'.")
print(f"   Final dataset shape: {final_synthetic_df.shape}")

✅ Successfully loaded original data from '../../data/data_with_features.csv'. Shape: (3786, 14)

⚙️  Starting grouped SARIMA generation using `statsmodels`...
   Found 224 unique time series to process.
   Processing group 1/224 (ID: T1R1)... Series too short, copying original data.
   Processing group 2/224 (ID: T1R10)... Series too short, copying original data.
   Processing group 3/224 (ID: T1R11)... Series too short, copying original data.
   Processing group 4/224 (ID: T1R12)... Series too short, copying original data.
   Processing group 5/224 (ID: T1R13)... Series too short, copying original data.
   Processing group 6/224 (ID: T1R14)... Series too short, copying original data.
   Processing group 7/224 (ID: T1R15)... Series too short, copying original data.
   Processing group 8/224 (ID: T1R16)... Series too short, copying original data.
   Processing group 9/224 (ID: T1R17)... Series too short, copying original data.
   Processing group 10/224 (ID: T1R18)... Series too short, 

: 